<a href="https://colab.research.google.com/github/majitoogomezzz-byte/proyecto_final/blob/main/Codigo_proyecto_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install osmnx
!pip install networkx
!pip install matplotlib
!pip install scikit-learn
!pip install folium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.4/104.4 kB 4.7 MB/s eta 0:00:00


In [ ]:
import osmnx as ox
import networkx as nx
import numpy as np
import json
import csv
import matplotlib.pyplot as plt
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

from google.colab import files

In [ ]:

import osmnx as ox
import networkx as nx
import numpy as np
import json
import matplotlib.pyplot as plt
import pandas as pd
import time
import folium

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

from google.colab import files


# VARIABLES GLOBALES


grafo = None
puntos = None
electrolineras = None
nodos_puntos = None
nodos_electrolineras = None

lista_puntos = None
lista_electrolineras = None

dataset_global = None

frecuencia_electrolineras = []
frecuencia_corredores = {}
frecuencia_por_vehiculo = {}

registros_recorridos = []

_cache_dist_ruta = {}
_cache_dist_electrolinera = {}


# OPCION 0 - GENERAR JSON AUTOMATICAMENTE


def generar_json():
    """
    Genera automáticamente el archivo datos_electrolineras.json
    con los puntos de referencia y electrolineras del proyecto.
    Lo guarda en Colab y lo descarga.
    """

    datos = {
        "puntos_referencia": [
            {"nombre": "UIS Campus Central",        "lat": 7.1398,  "lon": -73.1227},
            {"nombre": "UIS Campus Florida",         "lat": 7.0671,  "lon": -73.0836},
            {"nombre": "UIS Guatiguara Piedecuesta", "lat": 6.9731,  "lon": -73.0522},
            {"nombre": "UIS Campus Bucarica",        "lat": 7.1167,  "lon": -73.1103},
            {"nombre": "CENFER",                     "lat": 7.1272,  "lon": -73.1198},
            {"nombre": "UNAB",                       "lat": 7.1133,  "lon": -73.1072},
            {"nombre": "UTS",                        "lat": 7.0989,  "lon": -73.1056},
            {"nombre": "UPB",                        "lat": 7.1205,  "lon": -73.1148},
            {"nombre": "PTAR Rio Frio",              "lat": 7.0825,  "lon": -73.1011},
            {"nombre": "Sede Recreacional Catay",    "lat": 7.0702,  "lon": -73.0891}
        ],
        "electrolineras": [
            {"nombre": "Homecenter",                      "lat": 7.1124,  "lon": -73.1089},
            {"nombre": "CC Quinta Etapa",                 "lat": 7.1056,  "lon": -73.1034},
            {"nombre": "CC Cacique",                      "lat": 7.1278,  "lon": -73.1189},
            {"nombre": "CC Canaveral",                    "lat": 7.0983,  "lon": -73.1045},
            {"nombre": "Terpel Piedecuesta",              "lat": 6.9876,  "lon": -73.0534},
            {"nombre": "Exito La Rosita",                 "lat": 7.0767,  "lon": -73.0923},
            {"nombre": "CC La Florida",                   "lat": 7.0634,  "lon": -73.0845},
            {"nombre": "Promotores del Oriente (Giron)",  "lat": 7.0712,  "lon": -73.1678}
        ]
    }

    nombre_archivo = "datos_electrolineras.json"

    with open(nombre_archivo, "w", encoding="utf-8") as f:
        json.dump(datos, f, ensure_ascii=False, indent=4)

    print("\n ____JSON GENERADO ____")
    print(f"Archivo: {nombre_archivo}")
    print(f"Puntos de referencia: {len(datos['puntos_referencia'])}")
    print(f"Electrolineras: {len(datos['electrolineras'])}")
    print("_________________\n")

    print(f"Archivo '{nombre_archivo}' guardado en Colab.")
    print("Puedes descargarlo desde el panel de archivos (ícono carpeta, izquierda).")


# OPCION 1 - INICIALIZAR


def inicializar():

    global grafo
    global puntos
    global electrolineras
    global nodos_puntos
    global nodos_electrolineras
    global lista_puntos
    global lista_electrolineras
    global frecuencia_electrolineras
    global frecuencia_corredores
    global frecuencia_por_vehiculo
    global registros_recorridos
    global _cache_dist_ruta
    global _cache_dist_electrolinera

    archivo_subido = files.upload()

    if not archivo_subido:
        print("Error: no se subió ningún archivo.")
        return

    nombre_archivo = list(archivo_subido.keys())[0]

    if not nombre_archivo.endswith(".json"):
        print("Error: el archivo debe tener extensión .json")
        return

    try:
        datos = json.load(open(nombre_archivo, encoding="utf-8"))
    except json.JSONDecodeError:
        print("Error: el archivo no es un JSON válido.")
        return

    if "puntos_referencia" not in datos or "electrolineras" not in datos:
        print("Error: el JSON no tiene las claves 'puntos_referencia' y 'electrolineras'.")
        return

    puntos = datos["puntos_referencia"]
    electrolineras = datos["electrolineras"]

    if len(puntos) == 0 or len(electrolineras) == 0:
        print("Error: el JSON no contiene puntos o electrolineras.")
        return

    print("Descargando grafos de los municipios del área metropolitana...")

    municipios = [
        "Bucaramanga, Colombia",
        "Floridablanca, Colombia",
        "Girón, Colombia",
        "Piedecuesta, Colombia"
    ]

    grafos_municipios = [
        ox.graph_from_place(lugar, network_type="drive")
        for lugar in municipios
    ]

    grafo = nx.compose_all(grafos_municipios)

    def obtener_nodos(lista):
        nodos = {}
        for item in lista:
            nombre = item["nombre"]
            lat = item["lat"]
            lon = item["lon"]
            nodo = ox.distance.nearest_nodes(grafo, lon, lat)
            nodos[nombre] = nodo
        return nodos

    nodos_puntos        = obtener_nodos(puntos)
    nodos_electrolineras = obtener_nodos(electrolineras)

    lista_puntos        = [p["nombre"] for p in puntos]
    lista_electrolineras = [e["nombre"] for e in electrolineras]

    frecuencia_electrolineras = [0] * len(electrolineras)
    frecuencia_corredores     = {}
    frecuencia_por_vehiculo   = {}
    registros_recorridos      = []
    _cache_dist_ruta          = {}
    _cache_dist_electrolinera = {}

    print("Inicialización completada correctamente.")


# FUNCIONES AUXILIARES


def _dist_ruta(nombre_origen, nombre_destino):

    clave = (nombre_origen, nombre_destino)

    if clave not in _cache_dist_ruta:
        try:
            distancia = nx.shortest_path_length(
                grafo,
                nodos_puntos[nombre_origen],
                nodos_puntos[nombre_destino],
                weight="length"
            ) / 1000
        except (nx.NetworkXNoPath, nx.NodeNotFound):
            distancia = 0

        _cache_dist_ruta[clave] = distancia

    return _cache_dist_ruta[clave]


def _dist_electrolinera_desde(nodo_destino, nombre_destino):

    if nombre_destino in _cache_dist_electrolinera:
        return _cache_dist_electrolinera[nombre_destino]

    mejor_distancia = 999999
    mejor_indice    = 0

    for indice, nombre in enumerate(lista_electrolineras):
        try:
            distancia = nx.shortest_path_length(
                grafo,
                nodo_destino,
                nodos_electrolineras[nombre],
                weight="length"
            ) / 1000
        except (nx.NetworkXNoPath, nx.NodeNotFound):
            distancia = 9999

        if distancia < mejor_distancia:
            mejor_distancia = distancia
            mejor_indice    = indice

    _cache_dist_electrolinera[nombre_destino] = (mejor_distancia, mejor_indice)

    return mejor_distancia, mejor_indice

# OPCION 2 - SIMULAR

def simular():

    global registros_recorridos
    global frecuencia_electrolineras
    global frecuencia_corredores
    global frecuencia_por_vehiculo

    if grafo is None:
        print("Error: debes inicializar primero (opción 1).")
        return

    modelos = [
        (175, 29.8),
        (395, 116)
    ]

    nombres_modelos = [
        "Citroen e-C3",
        "Mercedes Benz G580"
    ]

    # VALIDAR SELECCIÓN DE VEHÍCULO
    while True:
        opcion_modelo = input(
            "Selecciona vehículo:\n  1. Citroen e-C3 (baja gama)\n  2. Mercedes Benz G580 (alta gama)\n> "
        ).strip()

        if opcion_modelo == "":
            print("Error: no puedes dejar este campo vacío.")
            continue

        if not opcion_modelo.isdigit():
            print("Error: ingresa solo números, sin letras ni símbolos.")
            continue

        if opcion_modelo not in ["1", "2"]:
            print("Error: opción fuera de rango. Ingresa 1 o 2.")
            continue

        seleccion = int(opcion_modelo) - 1
        break

    autonomia_km, bateria_total_kwh = modelos[seleccion]
    nombre_modelo = nombres_modelos[seleccion]
    consumo_por_km = bateria_total_kwh / autonomia_km

    # VALIDAR CANTIDAD DE RECORRIDOS
    while True:
        entrada = input("¿Cuántos recorridos deseas simular? > ").strip()

        if entrada == "":
            print("Error: no puedes dejar este campo vacío.")
            continue

        if entrada.startswith("-"):
            print("Error: no se permiten números negativos.")
            continue

        if not entrada.isdigit():
            print("Error: ingresa solo un número entero positivo, sin letras ni símbolos.")
            continue

        total_recorridos = int(entrada)

        if total_recorridos == 0:
            print("Error: debe ser al menos 1 recorrido.")
            continue

        if total_recorridos > 10000:
            print("Error: valor fuera de rango. Máximo permitido: 10000 recorridos.")
            continue

        break

    # Inicializar frecuencia por vehículo si no existe
    if nombre_modelo not in frecuencia_por_vehiculo:
        frecuencia_por_vehiculo[nombre_modelo] = [0] * len(electrolineras)

    bateria_actual = bateria_total_kwh

    for _ in range(total_recorridos):

        idx_origen, idx_destino = np.random.choice(
            len(lista_puntos), 2, replace=False
        )

        nombre_origen  = lista_puntos[idx_origen]
        nombre_destino = lista_puntos[idx_destino]

        corredor = f"{nombre_origen} - {nombre_destino}"

        if corredor not in frecuencia_corredores:
            frecuencia_corredores[corredor] = 0

        nodo_origen  = nodos_puntos[nombre_origen]
        nodo_destino = nodos_puntos[nombre_destino]

        bateria_inicio_pct = round(bateria_actual / bateria_total_kwh * 100, 2)

        camino = []

        if nx.has_path(grafo, nodo_origen, nodo_destino):

            camino = nx.shortest_path(
                grafo, nodo_origen, nodo_destino, weight="length"
            )

            for nodo_a, nodo_b in zip(camino, camino[1:]):
                arista = grafo.get_edge_data(nodo_a, nodo_b, 0)
                distancia_km = arista["length"] / 1000
                bateria_actual -= distancia_km * consumo_por_km
                bateria_actual = max(0, bateria_actual)

        bateria_fin_pct = round(bateria_actual / bateria_total_kwh * 100, 2)

        necesita_recarga = int(bateria_fin_pct <= 20)

        distancia_ruta_km = _dist_ruta(nombre_origen, nombre_destino)

        dist_min_km, indice_electrolinera = _dist_electrolinera_desde(
            nodo_destino, nombre_destino
        )

        if necesita_recarga:
            frecuencia_corredores[corredor] += 1
            frecuencia_electrolineras[indice_electrolinera] += 1
            frecuencia_por_vehiculo[nombre_modelo][indice_electrolinera] += 1
            bateria_actual = bateria_total_kwh * 0.8

        lat_origen  = next(i for i in puntos if i["nombre"] == nombre_origen)["lat"]
        lat_destino = next(i for i in puntos if i["nombre"] == nombre_destino)["lat"]
        lon_origen  = next(i for i in puntos if i["nombre"] == nombre_origen)["lon"]
        lon_destino = next(i for i in puntos if i["nombre"] == nombre_destino)["lon"]

        registros_recorridos.append({
            "corredor":                corredor,
            "origen":                  nombre_origen,
            "destino":                 nombre_destino,
            "modelo":                  nombre_modelo,
            "latitud_midpoint":        round((lat_origen + lat_destino) / 2, 6),
            "longitud_midpoint":       round((lon_origen + lon_destino) / 2, 6),
            "bateria_inicio_pct":      bateria_inicio_pct,
            "bateria_fin_pct":         bateria_fin_pct,
            "distancia_ruta_km":       round(distancia_ruta_km, 3),
            "consumo_estimado_kwh":    round(distancia_ruta_km * consumo_por_km, 4),
            "dist_electrolinera_min_km": round(dist_min_km, 3),
            "electrolinera_asignada":  lista_electrolineras[indice_electrolinera],
            "necesita_electrolinera":  necesita_recarga,
            "ruta":                    camino,
        })

    print(f"\nSimulación completada. Total registros: {len(registros_recorridos)}")


# OPCION 3 - GENERAR DATASET
def generar_dataset():

    global dataset_global

    if len(registros_recorridos) == 0:
        print("Error: primero debes simular recorridos (opción 2).")
        return

    dataset_global = pd.DataFrame(registros_recorridos)

    print("\n____ DATASET GENERADO ____")

    columnas_mostrar = [
        "corredor",
        "modelo",
        "distancia_ruta_km",
        "bateria_inicio_pct",
        "bateria_fin_pct",
        "electrolinera_asignada",
        "necesita_electrolinera"
    ]

    print(dataset_global[columnas_mostrar].to_string())
    print(f"\nFilas totales: {len(dataset_global)}")

    dataset_global.to_csv("dataset_recorridos.csv", index=False)
    files.download("dataset_recorridos.csv")

    print("\nDataset guardado y descargado como: dataset_recorridos.csv")
    print("_______________________\n")



# OPCION 4 - ENTRENAR IA


def entrenar_modelo():

    global dataset_global

    if dataset_global is None:
        print("Error: primero genera el dataset (opción 3).")
        return

    df = dataset_global

    X = df[[
        "distancia_ruta_km",
        "consumo_estimado_kwh",
        "dist_electrolinera_min_km"
    ]]

    y = df["necesita_electrolinera"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    modelo = RandomForestClassifier(n_estimators=100, random_state=42)

    #  MÉTRICA DE VELOCIDAD
    inicio = time.time()
    modelo.fit(X_train, y_train)
    fin = time.time()
    tiempo_entrenamiento = round(fin - inicio, 4)

    inicio_pred = time.time()
    predicciones = modelo.predict(X_test)
    fin_pred = time.time()
    tiempo_prediccion = round(fin_pred - inicio_pred, 4)

    precision = accuracy_score(y_test, predicciones)

    print("\n____ RESULTADOS IA ____")
    print(f"Variables usadas: {X.columns.tolist()}")
    print(f"Muestras entrenamiento: {len(X_train)} | Prueba: {len(X_test)}")
    print(f"\nPrecisión del modelo (accuracy): {round(precision * 100, 2)}%")
    print(f"Tiempo de entrenamiento:         {tiempo_entrenamiento}s")
    print(f"Tiempo de predicción:            {tiempo_prediccion}s")
    print("\nReporte detallado:")
    print(classification_report(y_test, predicciones,
                                 target_names=["No recarga", "Sí recarga"]))
    print("________________________\n")


# OPCION 5 - GRAFICAR


def graficar():

    if grafo is None:
        print("Error: debes inicializar primero (opción 1).")
        return

    if len(registros_recorridos) == 0:
        print("Error: primero simula recorridos (opción 2).")
        return

    fig, ax = ox.plot_graph(
        grafo,
        node_size=0,
        edge_color="#999999",
        edge_linewidth=0.5,
        bgcolor="white",
        show=False,
        close=False
    )

    for registro in registros_recorridos:
        ruta = registro["ruta"]
        if len(ruta) > 0:
            color_ruta = "red" if registro["necesita_electrolinera"] == 0 else "orange"
            ox.plot_graph_route(
                grafo, ruta,
                route_color=color_ruta,
                route_linewidth=2,
                node_size=0,
                orig_dest_size=0,
                show=False,
                close=False,
                ax=ax
            )

    for item in puntos:
        ax.scatter(item["lon"], item["lat"], c="blue", s=40, zorder=5)
        ax.text(item["lon"], item["lat"], item["nombre"], fontsize=7)

    for item in electrolineras:
        ax.scatter(item["lon"], item["lat"], c="green", s=80, marker="^", zorder=5)

    print("\n____ MAPA DE RECORRIDOS____")
    print(f"Total recorridos graficados: {len(registros_recorridos)}")
    print("Rojo:    recorrido normal")
    print("Naranja: necesitó recarga")
    print("Azul:    puntos de referencia")
    print("Verde:   electrolineras existentes")
    print("____________________________")

    plt.show()


# OPCION 6 - ANALISIS DE NUEVAS ELECTROLINERAS


def analizar_nuevas_electrolineras():
    """
    Analiza la frecuencia de uso de las electrolineras existentes
    y de los corredores con mayor demanda para recomendar
    dónde instalar nuevas electrolineras.
    """

    if len(registros_recorridos) == 0:
        print("Error: primero debes simular recorridos (opción 2).")
        return

    total_recargas = sum(frecuencia_electrolineras)

    if total_recargas == 0:
        print("No se registraron recargas en la simulación.")
        return

    print("\n____ ANÁLISIS DE NUEVAS ELECTROLINERAS ____")

    # 1. RANKING DE ELECTROLINERAS MÁS USADAS
    print("\n--- Uso de electrolineras existentes ---")
    print(f"{'#':<4} {'Electrolinera':<40} {'Recargas':>8} {'% del total':>12}")
    print("-" * 68)

    ranking = sorted(
        enumerate(frecuencia_electrolineras),
        key=lambda x: x[1],
        reverse=True
    )

    for pos, (idx, freq) in enumerate(ranking, 1):
        nombre = lista_electrolineras[idx]
        pct    = round(freq / total_recargas * 100, 1) if total_recargas > 0 else 0
        print(f"{pos:<4} {nombre:<40} {freq:>8} {pct:>11}%")

    # 2. FRECUENCIA POR TIPO DE VEHÍCULO
    if frecuencia_por_vehiculo:
        print("\n--- Recargas por tipo de vehículo ---")
        for vehiculo, freqs in frecuencia_por_vehiculo.items():
            total_v = sum(freqs)
            print(f"\n  Vehículo: {vehiculo} ({total_v} recargas)")
            for idx, freq in enumerate(freqs):
                if freq > 0:
                    print(f"    · {lista_electrolineras[idx]}: {freq} recargas")

    # 3. CORREDORES CON MAYOR DEMANDA DE RECARGA
    print("\n--- Top 5 corredores con más recargas ---")
    print(f"{'Corredor':<60} {'Recargas':>8}")
    print("-" * 70)

    corredores_ordenados = sorted(
        frecuencia_corredores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    top5 = [c for c in corredores_ordenados if c[1] > 0][:5]

    if not top5:
        print("  Ningún corredor registró recargas.")
    else:
        for corredor, freq in top5:
            print(f"{corredor:<60} {freq:>8}")

    #  4. RECOMENDACIÓN DE NUEVAS UBICACIONEs
    print("\n--- Recomendación de nuevas electrolineras ---")

    # Electrolineras con alta saturación (>30% del total)
    saturadas = [
        (lista_electrolineras[idx], freq)
        for idx, freq in ranking
        if freq / total_recargas > 0.30
    ]

    # Midpoints de los corredores más demandados como zonas candidatas
    zonas_candidatas = []
    for corredor, freq in top5:
        origen, destino = corredor.split(" - ")
        lat_o = next(p for p in puntos if p["nombre"] == origen)["lat"]
        lon_o = next(p for p in puntos if p["nombre"] == origen)["lon"]
        lat_d = next(p for p in puntos if p["nombre"] == destino)["lat"]
        lon_d = next(p for p in puntos if p["nombre"] == destino)["lon"]
        zonas_candidatas.append({
            "corredor": corredor,
            "recargas": freq,
            "lat_sugerida": round((lat_o + lat_d) / 2, 6),
            "lon_sugerida": round((lon_o + lon_d) / 2, 6)
        })

    if saturadas:
        print("\n  Electrolineras con alta saturación (>30% de recargas):")
        for nombre, freq in saturadas:
            print(f"     {nombre}: {freq} recargas — se recomienda refuerzo cercano")

    if zonas_candidatas:
        print("\n  Zonas recomendadas para nuevas electrolineras")
        print("  (basado en midpoint de corredores con mayor demanda):\n")
        print(f"  {'Corredor':<55} {'Lat':>10} {'Lon':>10} {'Recargas':>8}")
        print("  " + "-" * 85)
        for z in zonas_candidatas:
            print(f"  {z['corredor']:<55} {z['lat_sugerida']:>10} {z['lon_sugerida']:>10} {z['recargas']:>8}")

    # 5. GUARDAR ANÁLISIS EN JSON
    resultado = {
        "total_recargas": total_recargas,
        "uso_electrolineras": [
            {
                "nombre":   lista_electrolineras[idx],
                "recargas": freq,
                "porcentaje": round(freq / total_recargas * 100, 1)
            }
            for idx, freq in ranking
        ],
        "top5_corredores": [
            {"corredor": c, "recargas": f} for c, f in top5
        ],
        "zonas_nuevas_electrolineras": zonas_candidatas
    }

    with open("analisis_electrolineras.json", "w", encoding="utf-8") as f:
        json.dump(resultado, f, ensure_ascii=False, indent=4)

    files.download("analisis_electrolineras.json")

    print("\n  Análisis guardado y descargado como: analisis_electrolineras.json")
    print("__________________________________________________________\n")


# MENÚ


ACCIONES = {
    0: generar_json,
    1: inicializar,
    2: simular,
    3: generar_dataset,
    4: entrenar_modelo,
    5: graficar,
    6: analizar_nuevas_electrolineras
}

# BUCLE PRINCIPAL

while True:

    opcion = input(
        """
____________________________________________
  SISTEMA DE ELECTROLINERAS - BUCARAMANGA
____________________________________________
  0. Generar JSON automáticamente
  1. Inicializar
  2. Simular recorridos
  3. Generar dataset
  4. Entrenar IA
  5. Graficar mapa
  6. Analizar nuevas electrolineras
  7. Salir
____________________________________________
> """
    ).strip()

    if opcion == "":
        print("Error: debes ingresar una opción.")
        continue

    if not opcion.isdigit():
        print("Error: ingresa solo números, sin letras ni símbolos.")
        continue

    opcion = int(opcion)

    if opcion < 0 or opcion > 7:
        print("Error: opción fuera de rango. Ingresa un número entre 0 y 7.")
        continue

    if opcion == 7:
        print("Saliendo del sistema. ¡Hasta luego!")
        break

    accion = ACCIONES.get(opcion)

    if accion:
        accion()
    else:
        print("Opción no válida.")


  SISTEMA DE ELECTROLINERAS - BUCARAMANGA
  0. Generar JSON automáticamente
  1. Inicializar
  2. Simular recorridos
  3. Generar dataset
  4. Entrenar IA
  5. Graficar mapa
  6. Analizar nuevas electrolineras
  7. Salir
> 0

===== JSON GENERADO =====
Archivo: datos_electrolineras.json
Puntos de referencia: 10
Electrolineras: 8

Archivo 'datos_electrolineras.json' guardado en Colab.
Puedes descargarlo desde el panel de archivos (ícono carpeta, izquierda).
